<a href="https://colab.research.google.com/github/catacg/BDS-book/blob/master/Required_Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Required Task 4

Load the file financial_news.csv.

1. Last part of the sentence in each row of text contains an url. Remove this from text and create new column called URL and add the url.

2. Create sentence embeddings for the modified column text. Using Gradio, build a semantic search tool where the user enters some text (such as (“earnings surprise”, “regulatory fine”), and the top 5 closest records (based on cosine similarity) are output to the user.```



In [1]:
import pandas as pd

# Load the financial_news.csv file into a pandas DataFrame
df = pd.read_csv('/content/financial_news.csv')

In [2]:
# Display the first 5 rows of the DataFrame
display(df.head())

# Print information about the DataFrame, including column names, non-null counts, and data types
print(df.info())

,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16990 entries, 0 to 16989
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    16990 non-null  object
 1   label   16990 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 265.6+ KB
None


In [3]:
# Get the value counts of the 'label' column
label_counts = df['label'].value_counts()

# Display the label counts
print("Distribution of 'label' column:")
print(label_counts)

Distribution of 'label' column:
label
2     3545
18    2118
14    1822
9     1557
5      987
16     985
1      837
19     823
7      624
6      524
15     501
17     495
12     487
13     471
4      359
3      321
0      255
8      166
10      69
11      44
Name: count, dtype: int64


In [4]:
import re

# Define a regex pattern for URLs
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

def extract_and_remove_urls(text):
    urls = re.findall(url_pattern, text)
    # Remove the URLs from the text
    clean_text = re.sub(url_pattern, '', text).strip()
    # Return the cleaned text and the URLs joined by commas (or None if empty)
    url_string = ', '.join(urls) if urls else None
    return clean_text, url_string

# Apply the function to the DataFrame
# We'll use a temporary result column to split the outputs
results = df['text'].apply(lambda x: extract_and_remove_urls(str(x)))
df['text'] = results.apply(lambda x: x[0])

# Insert the URL column next to 'text'
# Since 'text' is at index 0, 'URL' will be at index 1
df.insert(1, 'URL', results.apply(lambda x: x[1]))

# Display rows where a URL was actually found to verify
display(df[df['URL'].notnull()].head())

,text,URL,label
0,Here are Thursday's biggest analyst calls: App...,https://t.co/QPN8Gwl7Uh,0
1,Buy Las Vegas Sands as travel to Singapore bui...,https://t.co/fLS2w57iCz,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",https://t.co/1EmtywmYpr,0
3,"Analysts react to Tesla's latest earnings, bre...",https://t.co/kwhoE6W06u,0
4,Netflix and its peers are set for a ‘return to...,https://t.co/jPpdl0D9s4,0


In [5]:
!pip install sentence-transformers gradio

In [6]:
from sentence_transformers import SentenceTransformer, util
import torch
import gradio as gr

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the 'text' column
print("Generating embeddings... this may take a minute.")
corpus_embeddings = model.encode(df['text'].tolist(), convert_to_tensor=True)
print("Embeddings generated successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings... this may take a minute.
Embeddings generated successfully.


In [ ]:
# Using a higher-performing model often yields better results for specialized terminology
# 'all-mpnet-base-v2' is generally more accurate than 'all-MiniLM-L6-v2'
print('Loading a more powerful model...')
better_model = SentenceTransformer('all-mpnet-base-v2')

print('Generating higher-quality embeddings...')
corpus_embeddings_better = better_model.encode(df['text'].tolist(), convert_to_tensor=True, show_progress_bar=True)
print('Done.')

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

print('Loading FinBERT model...')
finbert_name = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(finbert_name)
finbert_model = AutoModel.from_pretrained(finbert_name)

def get_finbert_embeddings(texts):
    # Mean Pooling - Take attention mask into account for correct averaging
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)
    with torch.no_grad():
        model_output = finbert_model(**encoded_input)

    # Use the mean of the last hidden states as the embedding
    token_embeddings = model_output[0]
    input_mask_expanded = encoded_input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

print('Generating FinBERT embeddings (this may take a moment)...')
# Processing in batches to avoid memory issues
all_texts = df['text'].tolist()
batch_size = 32
corpus_embeddings_finbert = []

for i in range(0, len(all_texts), batch_size):
    batch_texts = all_texts[i:i+batch_size]
    batch_emb = get_finbert_embeddings(batch_texts)
    corpus_embeddings_finbert.append(batch_emb)

corpus_embeddings_finbert = torch.cat(corpus_embeddings_finbert, dim=0)
print('FinBERT embeddings ready.')

In [ ]:
def compare_search(query):
    # 1. Original Model Search (MiniLM)
    query_emb_orig = model.encode(query, convert_to_tensor=True)
    scores_orig = util.cos_sim(query_emb_orig, corpus_embeddings)[0]
    top_orig = torch.topk(scores_orig, k=3)

    # 2. FinBERT Model Search
    query_emb_fin = get_finbert_embeddings([query])
    scores_fin = util.cos_sim(query_emb_fin, corpus_embeddings_finbert)[0]
    top_fin = torch.topk(scores_fin, k=3)

    output = {
        "Original (MiniLM) Results": [
            {"Text": df.iloc[idx.item()]['text'], "Score": f"{score.item():.4f}"}
            for score, idx in zip(top_orig[0], top_orig[1])
        ],
        "FinBERT Results": [
            {"Text": df.iloc[idx.item()]['text'], "Score": f"{score.item():.4f}"}
            for score, idx in zip(top_fin[0], top_fin[1])
        ]
    }
    return output

# Launch comparison interface
compare_interface = gr.Interface(
    fn=compare_search,
    inputs=gr.Textbox(label="Enter financial query"),
    outputs=gr.JSON(label="Model Comparison"),
    title="Model Comparison: MiniLM vs FinBERT"
)
compare_interface.launch(share=True)

In [ ]:
def semantic_search_comparison(query):
    # 1. Original MiniLM
    query_emb_orig = model.encode(query, convert_to_tensor=True)
    scores_orig = util.cos_sim(query_emb_orig, corpus_embeddings)[0]
    top_orig = torch.topk(scores_orig, k=5)

    res_orig = [[df.iloc[idx.item()]['text'], f"{score.item():.4f}"] for score, idx in zip(top_orig[0], top_orig[1])]

    # 2. FinBERT
    query_emb_fin = get_finbert_embeddings([query])
    scores_fin = util.cos_sim(query_emb_fin, corpus_embeddings_finbert)[0]
    top_fin = torch.topk(scores_fin, k=5)

    res_fin = [[df.iloc[idx.item()]['text'], f"{score.item():.4f}"] for score, idx in zip(top_fin[0], top_fin[1])]

    return res_orig, res_fin

with gr.Blocks(title="Financial Model Comparison") as demo:
    gr.Markdown("# Compare MiniLM vs FinBERT Search Results")
    with gr.Row():
        query_input = gr.Textbox(label="Enter Search Query", placeholder="e.g. regulatory fine")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Original (MiniLM-L6)")
            out_orig = gr.Dataframe(headers=["Text", "Score"], wrap=True)
        with gr.Column():
            gr.Markdown("### Specialized (FinBERT)")
            out_fin = gr.Dataframe(headers=["Text", "Score"], wrap=True)

    query_input.submit(semantic_search_comparison, inputs=query_input, outputs=[out_orig, out_fin])
    search_btn = gr.Button("Search")
    search_btn.click(semantic_search_comparison, inputs=query_input, outputs=[out_orig, out_fin])

demo.launch(share=True, inline=False)

### Interface Verification
Run the cell below to test if the `semantic_search_comparison` logic is responsive and returning results as expected.

In [ ]:
test_query = "regulatory fine"
try:
    mini_results, finbert_results = semantic_search_comparison(test_query)
    print(f"Verification Successful for query: '{test_query}'")
    print(f"MiniLM Top Result: {mini_results[0][0]} (Score: {mini_results[0][1]})")
    print(f"FinBERT Top Result: {finbert_results[0][0]} (Score: {finbert_results[0][1]})")
except Exception as e:
    print(f"Verification failed: {e}")

In [ ]:
# Run a test query to confirm the comparison interface logic
test_query = "earnings surprise"
print(f"--- Comparison Results for: '{test_query}' ---\n")

results_mini, results_finbert = semantic_search_comparison(test_query)

print("Top 3 MiniLM Results:")
for i, (text, score) in enumerate(results_mini[:3]):
    print(f"{i+1}. [Score: {score}] {text}")

print("\nTop 3 FinBERT Results:")
for i, (text, score) in enumerate(results_finbert[:3]):
    print(f"{i+1}. [Score: {score}] {text}")

In [ ]:
def semantic_search(query):
    # Encode the user query using the improved model
    query_embedding = better_model.encode(query, convert_to_tensor=True)

    # Compute cosine similarity between query and the better corpus embeddings
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings_better)[0]

    # Find the top 5 closest records
    top_results = torch.topk(cos_scores, k=5)

    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        row = df.iloc[idx.item()]
        results.append({
            "Text": row['text'],
            "URL": row['URL'],
            "Score": f"{score.item():.4f}"
        })

    return results

# Create Gradio Interface
interface = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(lines=2, placeholder="Enter search terms (e.g., 'earnings surprise', 'regulatory fine')..."),
    outputs=gr.JSON(label="Top 5 Closest Records (Using all-mpnet-base-v2)"),
    title="Improved Financial News Semantic Search",
    description="Search through financial news headlines using higher-quality semantic similarity."
)

# Launch with sharing enabled
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://60a09c07d746fae6f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
def semantic_search(query):
    # Encode the user query
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Compute cosine similarity between query and all corpus embeddings
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # Find the top 5 closest records
    top_results = torch.topk(cos_scores, k=5)

    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        row = df.iloc[idx.item()]
        results.append({
            "Text": row['text'],
            "URL": row['URL'],
            "Score": f"{score.item():.4f}"
        })
    return results

query = "earnings surprise"
search_results = semantic_search(query)

import json
print(f"Top 5 results for: '{query}'")
print(json.dumps(search_results, indent=2))

Top 5 results for: 'earnings surprise'
[
  {
    "Text": "SaaS Companies Earnings Are Coming: What To Expect?.   #stocks #trading #business",
    "URL": "https://t.co/6CxA8r9DdF",
    "Score": "0.6029"
  },
  {
    "Text": "Earnings season has begun! table from @eWhispers",
    "URL": "https://t.co/UpQJdKM22x",
    "Score": "0.5983"
  },
  {
    "Text": "@equitydd I expect a lot of volatility and wide ranges. Earnings reporting could pose some challenges.",
    "URL": null,
    "Score": "0.5731"
  },
  {
    "Text": "Wall Street Breakfast: Earnings Evaluation.   #markets #investing #stocks",
    "URL": "https://t.co/b8AD5CBj9s",
    "Score": "0.5675"
  },
  {
    "Text": "Finding the next surprising earnings winner like Netflix where expectations got too negative",
    "URL": "https://t.co/n1r0i7M0Fo",
    "Score": "0.5658"
  }
]
